# Case-loss ablation: matched GPU validation

Evaluate the frozen selected epoch-03 model on clean validation and the four medium / corruption-seed-586 conditions in `configs/ablation-case-loss.json`. Reuse the completed published/clean/augmented GPU results. One training seed; no held-out test and no training in this notebook.

Use the existing audited Colab T4 environment and the intended project checkout, including the new evaluation scripts/config. See `docs/case-loss-ablation-evaluation.md`. Do not Run all in the historical training notebook. Both execution switches below default to **False**.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT = Path('/content/cmpe586-turkish-punctuation')
STORAGE = Path('/content/drive/MyDrive/cmpe586-turkish-punctuation')
INPUTS = Path('/content/cmpe586-evaluation-inputs-noise-v1')
PREPARATION = STORAGE / 'preparation/case-loss-ablation-evaluation-v1/seed-586'
CONFIG = PROJECT / 'configs/ablation-case-loss-evaluation.json'
if not PROJECT.is_dir() or not CONFIG.is_file():
    raise RuntimeError('Restore the intended versioned project checkout first; do not use an older runner.')
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
ENV = {**os.environ, 'PYTHONPATH': str(PROJECT / 'src'),
       'TOKENIZERS_PARALLELISM': 'false', 'HF_DATASETS_DISABLE_PROGRESS_BARS': '1',
       'PYTHONUNBUFFERED': '1'}
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.used,memory.total',
                              '--format=csv,noheader'], text=True))
active = [line for line in subprocess.check_output(['ps', '-eo', 'pid,args'], text=True).splitlines()
          if any(name in line for name in ('scripts/train.py', 'scripts/evaluate_noise.py',
                                           'scripts/run_ablation_evaluation.py'))]
if active:
    raise RuntimeError(f'Another experiment is running: {active}')

def stream_command(command):
    import signal
    with subprocess.Popen(command, cwd=PROJECT, env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1,
                          start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            returncode = process.wait()
        except KeyboardInterrupt:
            os.killpg(process.pid, signal.SIGINT)
            process.wait()
            raise
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)


## Restore the existing audited validation bundle

The shared bundle contains validation only, including the previously prepared noise variants. Verify every member instead of regenerating noise. Existing copies are reused only after exact hash verification.


In [ ]:
import hashlib
import zipfile

def sha256(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

matrix = json.loads((PROJECT / 'configs/robustness-grid.json').read_text())
archive = STORAGE / 'inputs/validation-inputs-noise-v1.zip'
if sha256(archive) != matrix['input_bundle_sha256']:
    raise ValueError('Validation bundle hash differs.')
if not INPUTS.exists():
    stream_command([sys.executable, '-u', 'scripts/package_evaluation_inputs.py',
                    '--verify', str(archive), '--sha256', matrix['input_bundle_sha256'],
                    '--extract-to', str(INPUTS)])
else:
    with zipfile.ZipFile(archive) as packed:
        manifest = json.loads(packed.read('transfer-manifest.json'))
    if json.loads((INPUTS / 'transfer-manifest.json').read_text()) != manifest:
        raise ValueError('Restored manifest differs.')
    for relative, item in manifest['files'].items():
        target = INPUTS / relative
        if target.stat().st_size != item['bytes'] or sha256(target) != item['sha256']:
            raise ValueError(f'Changed restored input: {relative}')
    print('Verified existing validation inputs; no regeneration.')


## Technical readiness — disabled

Checks the frozen training identity, selected model files/tensors, original GPU calibration/runtime, numerical/data source hashes, exact reference commits and input manifest. Probes 16 fixed batches plus all planned longest-input batches in each of five conditions. No training updates or quality scores. A matching existing readiness receipt is reused without repeating probes.


In [ ]:
RUN_EVALUATION_PREFLIGHT = False
if RUN_EVALUATION_PREFLIGHT:
    RUN_EVALUATION_PREFLIGHT = False
    stream_command([sys.executable, '-u', 'scripts/prepare_ablation_evaluation.py',
                    '--input-root', str(INPUTS), '--storage-root', str(STORAGE),
                    '--output', str(PREPARATION / 'calibration.json')])
else:
    print('Evaluation preflight is OFF.')


## Five-condition validation evaluation — disabled

Readiness is independently verified first. The shared evaluator re-tokenizes all saved inputs, verifies unchanged word targets, matches batching and runs fresh clean + four noisy conditions. Verified completed conditions are retained on resume. A changed run identity stops execution.

After inference, the runner verifies 20 saved conditions against raw validation targets: 5 new ablation results and 15 existing results from the three fixed references. Comparison CSV/JSON/Markdown and a final hash commit are saved in the new run's `analysis/` directory. All checkpoints remain selected by clean validation.


In [ ]:
RUN_ABLATION_EVALUATION = False
if RUN_ABLATION_EVALUATION:
    RUN_ABLATION_EVALUATION = False
    stream_command([sys.executable, '-u', 'scripts/run_ablation_evaluation.py',
                    '--input-root', str(INPUTS), '--storage-root', str(STORAGE)])
else:
    print('Ablation evaluation is OFF.')


## Persistent results — read only

Read this after execution, or after reconnecting. The final analysis marker is written only after raw-target and source/input/model identity verification. This cell does not rerun inference.


In [ ]:
config = json.loads(CONFIG.read_text())
output = STORAGE / config['output_directory']
marker = output / 'analysis/complete.json'
if marker.exists():
    commit = json.loads(marker.read_text())
    for name, expected in commit['files'].items():
        if sha256(marker.parent / name) != expected:
            raise ValueError(f'Changed committed analysis: {name}')
    print((marker.parent / 'comparison.md').read_text())
    print('Completed and verified:', output)
else:
    status = output / 'progress.json'
    print(json.loads(status.read_text()) if status.exists() else 'No evaluation output yet.')
